# Channel audit: re-test the stricter failure-message prompt (v2) on the pilot

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arshia-HZ/mas-channel-audit/blob/main/notebooks/retest.ipynb)

The first pilot's failure messages were mostly vague (gate G4: 15% not vague). This notebook
re-tests a stricter prompt, `prompts/v2/failure_msg.txt`, on **copies of the finished pilot
shards**: it generates only the v2 failure messages and their revisions (about a fifth of the
pilot's work), validates each copy and uploads it under `pilot/<model>-v2`. The original
pilot shards are never changed, so both versions are compared on the same round 0 and pool.

**Needs** the pilot shards on Google Drive (or in the dataset, from which missing shards are
restored), `prompts/v2` **frozen** (the owner approves the wording and runs
`scripts/03_freeze_prompts.py --dir prompts/v2`), and an `HF_TOKEN` secret with write access.

When done it prints the pilot report with the new `failure_cause_v2` arm and prepares a
**fresh 40-message sample** (seed 1, different from the one already labelled) for gate G4,
with a format check (one sentence, no artifacts) and an audit of how many messages contain the
gold answer. Success criteria, set beforehand: at least half not vague, at least 95% single
sentence and no artifacts, at most one message containing the gold answer.

This is pilot data: it is never pooled into confirmatory results.

## 0. Settings

In [ ]:
MODEL_CONFIG = "configs/models/qwen3-1.7b.yaml"
BACKEND = "vllm"
REPLICATES = 2
SHARD_SIZE = 50
MAX_NEW_SHARDS = None  # e.g. 2 to stop this session after two new shards
UPLOAD = True  # push each finished shard to the private dataset
HF_REPO = "Arshia-HZ/mas-channel-audit-raw"
REF = "main"  # branch, tag or commit of the repository to run
REPO_URL = "https://github.com/Arshia-HZ/mas-channel-audit.git"
USE_DRIVE = True  # Colab only: the pilot shards live in Google Drive

## 1. Environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_COLAB:
    REPO = "/content/mas-channel-audit"
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        RUN_ROOT = "/content/drive/MyDrive/mas-channel-audit"
    else:
        RUN_ROOT = "/content/mas-channel-audit-runs"
elif IN_KAGGLE:
    REPO = "/kaggle/working/mas-channel-audit"
    RUN_ROOT = "/kaggle/working/runs"
else:  # a local checkout
    REPO = os.getcwd()
    if os.path.basename(REPO) == "notebooks":
        REPO = os.path.dirname(REPO)
    RUN_ROOT = os.environ.get("MCA_RUN_ROOT", os.path.join(REPO, "runs"))
os.makedirs(RUN_ROOT, exist_ok=True)


def sh(*cmd, cwd=None, check=True):
    """Run a command, show its output in the cell, raise if it fails."""
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(proc.stdout, end="")
    print(proc.stderr, end="", file=sys.stderr)
    if check and proc.returncode:
        raise RuntimeError(f"{' '.join(cmd)} exited with {proc.returncode}")
    return proc


print("colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local", "| repo:", REPO)
print("results go to:", RUN_ROOT)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    if not os.path.isdir(f"{REPO}/.git"):
        sh("git", "clone", "-q", REPO_URL, REPO)
    sh("git", "fetch", "-q", "--all", cwd=REPO)
    sh("git", "checkout", "-q", REF, cwd=REPO)
    if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO).returncode == 0:
        sh("git", "pull", "-q", "--ff-only", cwd=REPO)  # on a branch: take the latest
    extra = "vllm" if BACKEND == "vllm" else "hf"
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", f".[{extra},data]", cwd=REPO)

os.chdir(REPO)
sys.path.insert(
    0, f"{REPO}/src"
)  # a running kernel does not see a fresh editable install
sh("git", "log", "--oneline", "-1")

**Hugging Face token.** Needed to restore missing pilot shards and to upload the new ones
(write access). Store it as a Colab or Kaggle secret named `HF_TOKEN`; it is never printed.

In [ ]:
token = os.environ.get("HF_TOKEN")
if not token and IN_COLAB:
    from google.colab import userdata

    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
if not token and IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # the secret is simply not set
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found" if token else "not set (fine for the Qwen models)")

In [ ]:
import shutil
from importlib import metadata

if shutil.which("nvidia-smi"):
    sh("nvidia-smi", check=False)
for pkg in ("vllm", "torch", "transformers", "numpy", "math-verify", "pydantic"):
    try:
        print(f"{pkg:<14}{metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:<14}not installed")

## 2. Model, prompts and pilot problems

In [ ]:
from pathlib import Path

from channel_audit.models import load_model_config
from channel_audit.prompt_freeze import require_frozen

require_frozen(
    Path(REPO) / "prompts" / "v2"
)  # stops here until the owner has frozen v2

model = load_model_config(f"{REPO}/{MODEL_CONFIG}")
if model.backend != BACKEND:
    model = model.model_copy(update={"backend": BACKEND})
slug = model.model_id.split("/")[-1].lower()
SOURCE = f"{RUN_ROOT}/pilot_{slug}_{BACKEND}"
RUN = f"{SOURCE}_v2"
SOURCE_PREFIX = f"pilot/{slug}"
PREFIX = f"pilot/{slug}-v2"
print(model.model_id, "@", model.revision[:12], "|", model.backend, model.dtype)
print("source shards:", SOURCE)
print("v2 copies go to:", RUN, "| remote:", HF_REPO if UPLOAD else "(upload off)")

In [ ]:
from channel_audit.data import (
    MATH_REVISION,
    assign_splits,
    load_frozen,
    load_math_test,
    write_split,
)

SPLIT_DIR = os.environ.get("MCA_SPLIT_DIR", f"{REPO}/data/problems")
# Rebuilds the split from the pinned MATH revision and checks it against the committed
# manifest; refuses to continue if anything differs.
rows, dropped = load_math_test()
manifest = write_split(
    assign_splits(rows), SPLIT_DIR, n_rows=len(rows) + len(dropped), dropped=dropped
)
problems = load_frozen(SPLIT_DIR, "pilot")
print("MATH revision:", MATH_REVISION)
print(
    "pilot split:",
    len(problems),
    "problems, sha256",
    manifest["files"]["pilot"]["sha256"][:16],
)

## 3. Generate the v2 failure messages and revisions

The model is loaded **once**, on the first shard that needs it. Run all again after a
disconnect: finished copies are skipped.

In [ ]:
from channel_audit.runner import backend_for_run, config_for_model
from channel_audit.shards import (
    LazyBackend,
    hf_downloader,
    hf_uploader,
    make_shards,
    run_variant_shards,
)

cfg = config_for_model(model, replicates=REPLICATES, log_prompts=True)
uploader = restorer = None
if UPLOAD:
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("UPLOAD needs the HF_TOKEN secret (write access)")
    uploader = hf_uploader(HF_REPO, token)
    restorer = hf_downloader(HF_REPO, token)

n_shards = len(make_shards(problems, SHARD_SIZE))
results = run_variant_shards(
    SOURCE,
    RUN,
    cfg,
    problems,
    LazyBackend(lambda: backend_for_run(cfg, problems)),
    size=SHARD_SIZE,
    uploader=uploader,
    restorer=restorer,
    source_prefix=SOURCE_PREFIX,
    remote_prefix=PREFIX,
    max_new_shards=MAX_NEW_SHARDS,
)
for r in results:
    print(
        f"{r.name}: {r.status}, {r.n_generations} generations, "
        f"G3 {'pass' if r.g3_ok else 'FAIL'}, uploaded {r.uploaded}"
    )
shard_dirs = sorted(
    d for d in Path(RUN).glob("shard_*") if (d / "SHARD_DONE.json").exists()
)
ALL_DONE = len(shard_dirs) == n_shards
print(
    f"{len(shard_dirs)} of {n_shards} copies finished",
    "- run all again to continue" if not ALL_DONE else "",
)

## 4. Gates, with the new arm

`failure_cause_v2` appears next to `failure_cause` in the arm table and as one more contrast.
G4 is pending until the new sample is labelled (next section).

In [ ]:
import json

from channel_audit.gates import format_pilot_report, pilot_report
from channel_audit.runner import open_run

if ALL_DONE:
    ctxs = [open_run(d) for d in shard_dirs]
    report = pilot_report(ctxs, manifest["counts"]["confirm"])
    print(format_pilot_report(report, None))
    with open(f"{RUN}/pilot_report_v2.json", "w", encoding="utf-8") as fh:
        json.dump(report, fh, indent=1, sort_keys=True)
else:
    print("not all copies are finished yet")

## 5. A fresh sample of 40 v2 messages (gate G4)

Seed 1 and the v2 variant: a different draw from the one labelled earlier. Read
`failure_sheet.md`, fill each label in `labels.json` with `useful`, `vague` or `wrong`, and
send me both files.

In [ ]:
from channel_audit.failure_review import (
    format_label_sheet,
    format_stats,
    gold_audit,
    sample_failure_messages,
    write_label_template,
)

if ALL_DONE:
    G4_DIR = Path(RUN) / "g4_v2"
    G4_DIR.mkdir(exist_ok=True)
    items = sample_failure_messages(ctxs, n=40, seed=1, variant="v2")
    (G4_DIR / "failure_sheet.md").write_text(
        format_label_sheet(items), encoding="utf-8"
    )
    if not (G4_DIR / "labels.json").exists():
        write_label_template(items, G4_DIR / "labels.json")
    stats, audit = format_stats(items), gold_audit(items)
    print(
        f"{stats['single_sentence']}/{stats['n']} single sentence, "
        f"{stats['artifacts']} with an [answer removed] artifact, "
        f"{audit['contains_gold']} containing the gold answer"
    )
    if IN_COLAB:
        from google.colab import files

        for name in ("failure_sheet.md", "labels.json"):
            files.download(str(G4_DIR / name))
        files.download(f"{RUN}/pilot_report_v2.json")

**Send back:** the printed report, the format and gold-audit line, `pilot_report_v2.json` and the
filled `labels.json`.